In [29]:
cities = {
    "Miami FL": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=363&data_layer=scenario&format=csv",
    "New Orleans LA": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=2215&data_layer=scenario&format=csv",
    "Norfolk VA": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=399&data_layer=scenario&format=csv",
    "New York NY": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=12&data_layer=scenario&format=csv",
    "Atlantic City NJ": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=180&data_layer=scenario&format=csv",
    "Charleston SC": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=234&data_layer=scenario&format=csv",
    "Tampa FL": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=520&data_layer=scenario&format=csv",
    "Galveston TX": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=161&data_layer=scenario&format=csv",
    "Honolulu HI": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=155&data_layer=scenario&format=csv",
    "Seattle WA": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=127&data_layer=scenario&format=csv",
    "Boston MA": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=235&data_layer=scenario&format=csv",
    "San Francisco CA": "https://d3qt3aobtsas2h.cloudfront.net/edge/ws/search/projection?psmsl_id=10&data_layer=scenario&format=csv",
}

In [30]:
import pandas as pd
from io import BytesIO
import requests

all_dfs = []

for city, url in cities.items():

    response = requests.get(url)

    # read Excel file from memory
    xls = pd.ExcelFile(BytesIO(response.content))

    # use Total sheet
    df = pd.read_excel(xls, sheet_name="Total")

    # median projection only
    df = df[df["quantile"] == 50]

    # scenarios you use
    df = df[df["scenario"].isin([
        "ssp126",
        "ssp245",
        "ssp370",
        "ssp585"
    ])]

    # reshape years into rows
    year_cols = [c for c in df.columns if str(c).isdigit()]

    df_long = df.melt(
        id_vars=["scenario", "quantile"],
        value_vars=year_cols,
        var_name="year",
        value_name="total_sea_level_m"
    )

    df_long["year"] = df_long["year"].astype(int)

    # meters → cm
    df_long["total_sea_level_cm"] = (
        df_long["total_sea_level_m"] * 100
    )

    df_long["city"] = city

    all_dfs.append(
        df_long[
            [
                "city",
                "scenario",
                "year",
                "total_sea_level_cm"
            ]
        ]
    )

combined = pd.concat(all_dfs, ignore_index=True)
combined.to_csv(
    "data/all_cities_total_sea_level.csv",
    index=False
)

In [31]:
import pandas as pd

df = pd.read_csv("data/all_cities_total_sea_level.csv")

# make sure year is numeric
df["year"] = df["year"].astype(int)

all_groups = []

for (city, scenario), group in df.groupby(["city", "scenario"]):
    group = group.sort_values("year")

    # create yearly range from 2020 to 2100
    full_years = pd.DataFrame({
        "year": range(2020, 2101)
    })

    # merge decade data onto yearly rows
    merged = full_years.merge(group, on="year", how="left")

    merged["city"] = city
    merged["scenario"] = scenario

    # linearly interpolate missing yearly values
    merged["total_sea_level_cm"] = (
        merged["total_sea_level_cm"]
        .interpolate(method="linear")
    )

    all_groups.append(merged)

yearly = pd.concat(all_groups, ignore_index=True)

yearly = yearly[
    ["city", "scenario", "year", "total_sea_level_cm"]
]

yearly.to_csv("data/all_cities_total_sea_level.csv", index=False)

yearly.head()

,city,scenario,year,total_sea_level_cm
0,Atlantic City NJ,ssp126,2020,11.50
1,Atlantic City NJ,ssp126,2020,11.70
2,Atlantic City NJ,ssp126,2021,12.59
3,Atlantic City NJ,ssp126,2022,13.48
4,Atlantic City NJ,ssp126,2023,14.37
